In [1]:
# Gaussian Mixture Model (GMM) for Change Detection
# -------------------------------------------------
# Instructions: Below, I provide a code structure as a guidance to get you started. Of course you can use your own code structure to achieve the objective. Your marks will NOT be deducted if you use your own code structure :) 
# Complete the sections marked with '### ENTER YOUR CODE HERE ###' to implement 
# the Gaussian Mixture Model for background subtraction and change detection.
'''
useful resources:
https://moodle.bath.ac.uk/mod/forum/discuss.php?d=534501
https://ieeexplore.ieee.org/document/4723224
'''
import cv2
import numpy as np

# Initialize the GMMBackgroundSubtractor Class
class GMMBackgroundSubtractor:
    def __init__(self, frame_shape, num_gaussians= 4, learning_rate=0.001, threshold=2.5):
        """
        Initialize Gaussian parameters: means, variances, and weights.

        Args:
            frame_shape (tuple): Shape of the input frame (height, width).
            num_gaussians (int): Number of Gaussian models per pixel.
            learning_rate (float): Rate at which the model updates.
            threshold (float): Threshold for matching a pixel to a Gaussian.
        """
        self.num_gaussians = num_gaussians
        self.learning_rate = learning_rate
        self.threshold = threshold
        
        # Initialize the means, variances, and weights for each Gaussian
        self.means = np.random.randint(0, 256, (frame_shape[0], frame_shape[1], self.num_gaussians)).astype(np.float32)
        self.variances = np.full((frame_shape[0], frame_shape[1], self.num_gaussians), 10**2, dtype = np.float32)
        self.weights = np.full((frame_shape[0], frame_shape[1], self.num_gaussians), 1 / self.num_gaussians, dtype=np.float32)

        # Normalise the weights
        self.weights /= np.sum(self.weights, axis=2, keepdims=True)  
        self.weights = np.clip(self.weights, 0.01, 1)

        
        
    def apply(self, frame):
        """
        Apply GMM to detect foreground objects.

        Args:
            frame (ndarray): Input video frame.

        Returns:
            foreground_mask (ndarray): Binary mask indicating foreground pixels.
        """
        # Convert frame to grayscale if it isn't already
        if len(frame.shape) == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        
        frame = frame.astype(np.float32)
        
        # Expand frame to match gaussian dimensions
        pixel_values = np.repeat(frame[:, :, np.newaxis], self.num_gaussians, axis=2)
        
        # Calculate the absolute difference between the frame and Gaussian means
        abs_diff = np.abs(frame[:, : , np.newaxis] - self.means) # increase the dimension of the existing array

        # Check which pixels match any of the Gaussians
        matched = abs_diff < (self.threshold * np.sqrt(self.variances))
        
        # Create an empty foreground mask
        foreground_mask = np.ones(frame.shape, dtype=np.uint8) 

        # Step 1: Update matched Gaussians (means, variances, weights)
        
        matched_gaussian = np.argmax(matched, axis=2) 

        for i in range(self.num_gaussians):
            match = (matched_gaussian == i)
            self.means[match, i] = (1 - self.learning_rate) * self.means[match, i] + self.learning_rate * frame[match]
            self.variances[match, i] = (1 - self.learning_rate) * self.variances[match, i] + self.learning_rate * (frame[match] - self.means[match, i])**2
            self.weights[match, i] += self.learning_rate    

        # Reduce the weights of unmatched Gaussians
        self.weights[~matched] *= (1 - self.learning_rate * 2)
            
        # Normalize weights so they sum to 1
        self.weights /= np.sum(self.weights, axis=2, keepdims=True)
        self.weights = np.clip(self.weights, 0.01, 1)

        # Step 2: Classify Foreground
        best_fit = np.any(matched, axis=2)  
        foreground_mask[best_fit] = 0  
        
        return foreground_mask.astype(np.uint8) * 255

# ------------------- Main Code to Run the GMM ------------------- #

# Load Video
video_path = 'GMM_input-cars.mp4'
cap = cv2.VideoCapture(video_path)


# Read the first frame to get the frame shape
ret, frame = cap.read()

if not ret:
    print("Error: Could not read the video file")
    cap.release()
    exit()

# Initialize GMM Subtractor
gmm_subtractor = GMMBackgroundSubtractor(frame.shape[:2], num_gaussians=3, learning_rate=0.001, threshold=2.5)

# Process the video frame by frame
### ENTER YOUR CODE TO PROCESS THE VIDEO HERE ###
while True:  
    ret, frame = cap.read()
    if not ret:
        break
    
    # Apply GMM background subtraction to get the foreground mask
    foreground_mask = gmm_subtractor.apply(frame)
    
    # Clean the noise 
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (1, 1))

    # Morphological Operations
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_OPEN, kernel)
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_CLOSE, kernel)
    foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_CLOSE, kernel)
    foreground_mask = cv2.dilate(foreground_mask, kernel, iterations=2)
    #foreground_mask = cv2.GaussianBlur(foreground_mask, (5, 5), 0) #tried the guassian blur, makes the result worse
    #_, foreground_mask = cv2.threshold(foreground_mask, 127, 255, cv2.THRESH_BINARY)

    # Display the foreground mask
    cv2.imshow('Input', frame)
    cv2.imshow('Foreground Mask', foreground_mask)
    
    # Quit the video (press 'q')
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release resources and close windows
cap.release()
cv2.destroyAllWindows()